In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [ ]:
DATA = Path("/content")

matches     = pd.read_csv(DATA / "match.csv")
predictions = pd.read_csv(DATA / "match_prediction.csv")
users       = pd.read_csv(DATA / "users.csv")
winner      = pd.read_csv(DATA / "winner_prediction.csv")
country     = pd.read_csv(DATA / "country.csv")

In [ ]:
OUTPUT = Path("OutputData/PredictionWC")
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
def save_json(name, df):
    path = OUTPUT / f"{name}.json"
    with open(path, "w") as f:
        json.dump(df.where(pd.notnull(df), None).to_dict(orient="records"), f, indent=2)
    print(path)

In [ ]:
name_map = country.set_index("id")["name"]

matches = matches.dropna(subset=["goal1", "goal2"]).copy()
matches[["goal1", "goal2"]] = matches[["goal1", "goal2"]].astype(int)
matches["team1_name"] = matches["team1"].map(name_map)
matches["team2_name"] = matches["team2"].map(name_map)

merged = predictions.merge(
    matches[["id", "team1", "team2", "team1_name", "team2_name", "goal1", "goal2", "match_type"]],
    left_on="match_id", right_on="id", suffixes=("_pred", "_actual")
)
merged["error_team1"] = (merged.goal1_pred - merged.goal1_actual).abs()
merged["error_team2"] = (merged.goal2_pred - merged.goal2_actual).abs()
merged["total_error"] = merged.error_team1 + merged.error_team2

In [ ]:
def outcome(g1, g2):
    return np.select([g1 > g2, g1 < g2], ["team1", "team2"], default="draw")

merged["predicted_outcome"] = outcome(merged.goal1_pred, merged.goal2_pred)
merged["actual_outcome"]    = outcome(merged.goal1_actual, merged.goal2_actual)

grp = merged.groupby("match_id")

votes = (
    grp["predicted_outcome"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("pct")
    .reset_index()
)
consensus = (
    votes.loc[votes.groupby("match_id")["pct"].idxmax()]
    .rename(columns={"predicted_outcome": "consensusOutcome", "pct": "consensus"})
)
accuracy = (
    grp.apply(lambda g: (g.predicted_outcome == g.actual_outcome).mean() * 100)
    .rename("accuracy")
    .reset_index()
)
match_meta = merged[["match_id", "team1_name", "team2_name", "actual_outcome"]].drop_duplicates("match_id")
totals     = grp.size().rename("totalPredictions").reset_index()

conf = (
    consensus
    .merge(accuracy,    on="match_id")
    .merge(match_meta,  on="match_id")
    .merge(totals,      on="match_id")
)
conf["match"]         = conf.team1_name + " vs " + conf.team2_name
conf["confidenceGap"] = (conf.consensus - conf.accuracy).round(2)
conf["consensus"]     = conf.consensus.round(2)
conf["accuracy"]      = conf.accuracy.round(2)
conf = conf.sort_values("confidenceGap", ascending=False)

save_json("confidence_vs_accuracy", conf[[
    "match_id", "match", "consensusOutcome", "actual_outcome",
    "consensus", "accuracy", "confidenceGap", "totalPredictions"
]])

In [ ]:
home_preds = merged[["team1", "team1_name", "goal1_pred", "goal1_actual"]].rename(
    columns={"team1": "team_id", "team1_name": "team", "goal1_pred": "predicted_goals", "goal1_actual": "actual_goals"}
)
away_preds = merged[["team2", "team2_name", "goal2_pred", "goal2_actual"]].rename(
    columns={"team2": "team_id", "team2_name": "team", "goal2_pred": "predicted_goals", "goal2_actual": "actual_goals"}
)
team_predictions = pd.concat([home_preds, away_preds], ignore_index=True)

misjudged = (
    team_predictions
    .groupby(["team_id", "team"])
    .agg(
        predictedGoals=("predicted_goals", "mean"),
        actualGoals=("actual_goals", "mean"),
        matches=("team_id", "count")
    )
    .reset_index()
)
# positive = overestimated, negative = underestimated
misjudged["misjudgment"] = misjudged.predictedGoals - misjudged.actualGoals
misjudged = misjudged.sort_values("misjudgment", ascending=False)

save_json("most_misjudged_teams", misjudged)

In [ ]:
merged["exact"] = (
    (merged.goal1_pred == merged.goal1_actual) &
    (merged.goal2_pred == merged.goal2_actual)
)

lucky = (
    merged.groupby("user_id")
    .agg(exactScores=("exact", "sum"))
    .reset_index()
    .merge(users[["id", "full_name"]], left_on="user_id", right_on="id")
    .sort_values("exactScores", ascending=False)
)

save_json("luckiest_predictors", lucky[["user_id", "full_name", "exactScores"]])

In [ ]:
home = merged[["team1", "team1_name", "goal1_pred", "goal1_actual"]].rename(
    columns={"team1": "team", "team1_name": "team_name", "goal1_pred": "predicted", "goal1_actual": "actual"}
)
away = merged[["team2", "team2_name", "goal2_pred", "goal2_actual"]].rename(
    columns={"team2": "team", "team2_name": "team_name", "goal2_pred": "predicted", "goal2_actual": "actual"}
)
teamGoals = pd.concat([home, away], ignore_index=True)

biased = (
    teamGoals.groupby(["team", "team_name"])
    .agg(predictedGoals=("predicted", "mean"), actualGoals=("actual", "mean"))
    .reset_index()
)
biased["bias"] = biased.predictedGoals - biased.actualGoals
biased = biased.sort_values("bias", ascending=False)

save_json("biased_fan", biased[["team_name", "predictedGoals", "actualGoals", "bias"]])

In [ ]:
teamTotals = (
    teamGoals.groupby(["team", "team_name"])
    .agg(predicted=("predicted", "sum"), actual=("actual", "sum"))
    .reset_index()
)
teamTotals["difference"] = teamTotals.actual - teamTotals.predicted
teamTotals = teamTotals.sort_values("difference", ascending=False)

save_json("biggest_surprises", teamTotals[["team_name", "predicted", "actual", "difference"]])

In [ ]:
merged["predicted_team"] = np.where(
    merged.goal1_pred > merged.goal2_pred, merged.team1,
    np.where(merged.goal2_pred > merged.goal1_pred, merged.team2, np.nan)
)

trusted = (
    merged.dropna(subset=["predicted_team"])
    .assign(predicted_team=lambda d: d.predicted_team.astype(int))
    .groupby("predicted_team")
    .size()
    .reset_index(name="predictedWins")
)
trusted["name"]              = trusted.predicted_team.map(name_map)
trusted["predictionPercent"] = (trusted.predictedWins / trusted.predictedWins.sum() * 100).round(2)
trusted = trusted.sort_values("predictedWins", ascending=False)

save_json("most_trusted_teams", trusted[["name", "predictedWins", "predictionPercent"]])

In [ ]:
heatmap = (
    predictions.groupby(["goal1", "goal2"])
    .size()
    .reset_index(name="count")
    .sort_values(["goal1", "goal2"])
)

save_json("prediction_heatmap", heatmap)

In [ ]:
actual_heatmap = (
    matches.groupby(["goal1", "goal2"])
    .size()
    .reset_index(name="count")
    .sort_values(["goal1", "goal2"])
)

save_json("actual_heatmap", actual_heatmap)

In [ ]:
top_predicted = (
    predictions.groupby(["goal1", "goal2"])
    .size()
    .reset_index(name="timesPredicted")
    .sort_values("timesPredicted", ascending=False)
    .head(10)
)
actual_counts = (
    matches.groupby(["goal1", "goal2"])
    .size()
    .reset_index(name="timesOccurred")
)

score_psych = top_predicted.merge(actual_counts, on=["goal1", "goal2"], how="left")
score_psych["timesOccurred"] = score_psych["timesOccurred"].fillna(0).astype(int)
score_psych["score"] = score_psych.goal1.astype(str) + "-" + score_psych.goal2.astype(str)

save_json("score_psychology", score_psych[["score", "timesPredicted", "timesOccurred"]])

In [ ]:
stage_acc = (
    merged.groupby("match_type")
    .apply(lambda g: (g.predicted_outcome == g.actual_outcome).mean() * 100)
    .rename("accuracy")
    .round(2)
    .reset_index()
)

save_json("accuracy_by_stage", stage_acc)

In [ ]:
print(winner.columns.tolist())

winner_picks = (
    winner.groupby("country_id") 
    .size()
    .reset_index(name="picks")
)
winner_picks["name"]        = winner_picks["country_id"].map(name_map)
winner_picks["pickPercent"] = (winner_picks.picks / winner_picks.picks.sum() * 100).round(2)
winner_picks = winner_picks.sort_values("picks", ascending=False)

save_json("tournament_winner_predictions", winner_picks[["name", "picks", "pickPercent"]])